# Bölüm 9 — BÖLÜM 9: Metin Madenciliği ve Doğal Dil İşleme (NLP)

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 9. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q beautifulsoup4 gensim matplotlib nltk numpy pandas scikit-learn spacy torch transformers


## 9.1. Metin Ön İşleme ve Temel Temsil


### 9.1.1.1. Ham Metin Temizleme

`bolum09/09_01_01_01_ham-metin-temizleme.py`


In [ ]:
import re
from bs4 import BeautifulSoup

def ham_metin_temizle(metin):
    """
    Ham metin gürültüsünü kademeli olarak temizleyen fonksiyon.
    """
    # 1. HTML etiketlerini kaldır
    metin = BeautifulSoup(metin, 'html.parser').get_text()
    # 2. URL'leri kaldır (http, https, www ile başlayanlar)
    metin = re.sub(r'http\S+|www\.\S+', '', metin)
    # 3. Sosyal medya etiketleri: @mention ve #hashtag
    metin = re.sub(r'@\w+|#\w+', '', metin)
    # 4. Özel karakterleri ve noktalama işaretlerini kaldır
    metin = re.sub(r'[^a-zA-ZğüşöçıİĞÜŞÖÇ\s]', ' ', metin)
    # 5. Küçük harfe çevir
    metin = metin.lower()
    # 6. Fazla boşlukları tek boşluğa indir
    metin = re.sub(r'\s+', ' ', metin).strip()
    return metin

# Test
ornek = '<p>Bu ürün MÜKEMMEL! https://shop.com/urun @kullanici #indirim fiyat: 299₺</p>'
print('Orijinal :', ornek)
print('Temizlenmiş:', ham_metin_temizle(ornek))
# Çıktı: 'bu ürün mükemmel fiyat'


### Tokenization Türleri

`bolum09/09_01_01_02_tokenization-turleri.py`

_Kitap: Kod 9.1_


In [ ]:
import nltk
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer
from nltk.util import ngrams

# Gerekli NLTK verileri (ilk kullanımda indirilmeli)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

metin = "Doğal dil işleme, yapay zekanın en heyecan verici dallarından biridir! "\
         "ChatGPT, BERT ve GPT-4 gibi modeller bu alanı kökten değiştirdi."

# 1. Kelime Tokenization (NLTK)
print('=== NLTK Kelime Tokenization ===')
nltk_tokens = word_tokenize(metin)
print(nltk_tokens)
# ['Doğal', 'dil', 'işleme', ',', 'yapay', 'zekanın', ...]

# 2. Cümle Tokenization (NLTK)
print('\n=== NLTK Cümle Tokenization ===')
cumleler = sent_tokenize(metin)
for i, c in enumerate(cumleler):
    print(f'Cümle {i+1}: {c}')

# 3. Tweet Tokenizer (Sosyal Medya için özelleştirilmiş)
tweet = "Harika ürün!! :) @marka #indirim fiyat 299 TL http://link.com"
tweet_tokenizer = TweetTokenizer(strip_handles=True, reduce_len=True)
print('\n=== Tweet Tokenizer ===')
print(tweet_tokenizer.tokenize(tweet))
# ['Harika', 'ürün', '!', ':)', '#indirim', 'fiyat', '299', 'TL']

# 4. N-gram Tokenization
print('\n=== Bigram (2-gram) Tokenization ===')
bigrams = list(ngrams(word_tokenize('veri madenciliği çok önemli bir alandır'), 2))
print(bigrams[:5])

# 5. spaCy ile Gelişmiş Tokenization (POS ve bağımlılık bilgisiyle birlikte)
try:
    nlp = spacy.load('en_core_web_sm')
    doc = nlp('Data mining and NLP are key topics in AI research.')
    print('\n=== spaCy Tokenization (POS etiketleriyle) ===')
    for token in doc:
        print(f'{token.text:<15} POS: {token.pos_:<8} DEP: {token.dep_}')
except OSError:
    print('spaCy modeli yüklenmedi. python -m spacy download en_core_web_sm')


### 9.1.1.3. Durdurma Kelimeleri (Stop-words) Temizliği

`bolum09/09_01_01_03_durdurma-kelimeleri-temizligi.py`

_Kitap: Kod 9.2_


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# 1. Mevcut stop-words listeleri
ingilizce_sw = set(stopwords.words('english'))
print(f'İngilizce stop-words sayısı: {len(ingilizce_sw)}')
print(f'Örnek: {list(ingilizce_sw)[:10]}')

# Türkçe için manual liste veya özel kütüphane gerekebilir
turkce_stop_words = {
    'bir', 've', 'bu', 'da', 'de', 'ile', 'için', 'ama', 'veya',
    'ki', 'gibi', 'mi', 'mu', 'mü', 'mı', 'ne', 'ya', 'çok',
    'daha', 'en', 'bunu', 'buna', 'bunun', 'olan', 'olarak', 'olan'
}

# 2. Örnek metin
metin = """
Veri madenciliği ve makine öğrenmesi ile büyük veri setlerinden
anlamlı örüntüler çıkarmak için bu teknikleri kullanıyoruz.
Bu yöntemler çok güçlü ve etkili araçlardır.
"""

# 3. Tokenize et ve stop-words temizle
tokenlar = word_tokenize(metin.lower())
temiz_tokenlar = [t for t in tokenlar
                  if t.isalpha() and t not in turkce_stop_words]

print(f'\nOrijinal token sayısı: {len(tokenlar)}')
print(f'Temizlenmiş token sayısı: {len(temiz_tokenlar)}')
print(f'Temiz tokenlar: {temiz_tokenlar}')

# 4. En sık kelimeler karşılaştırması
print('\n=== Stop-words Öncesi En Sık 5 Kelime ===')
print(Counter([t for t in tokenlar if t.isalpha()]).most_common(5))

print('\n=== Stop-words Sonrası En Sık 5 Kelime ===')
print(Counter(temiz_tokenlar).most_common(5))

# 5. Alan özgü (domain-specific) stop-words ekleme
domain_stop_words = turkce_stop_words | {'yöntemler', 'araçlardır', 'kullanıyoruz'}
ultra_temiz = [t for t in tokenlar if t.isalpha() and t not in domain_stop_words]
print(f'\nAlan özgü temizlik sonrası: {ultra_temiz}')


### Lemmatization (Gövdeleme)

`bolum09/09_01_01_04_lemmatization.py`

_Kitap: Kod 9.3_


In [ ]:
import re
import nltk
import spacy
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

# Gerekli NLTK verileri
for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger',
                  'punkt_tab', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

# Örnek Metin
metin = "The quick brown foxes are jumping over the lazy dogs. Studies show they are better runners."

# 1. Ön Temizlik ve Tokenization
temiz = re.sub(r'[^a-zA-Z\s]', '', metin).lower()
tokenlar = word_tokenize(temiz)
stop_words = set(stopwords.words('english'))
temiz_tokenlar = [t for t in tokenlar if t not in stop_words]

print('Temizlenmiş Tokenlar:', temiz_tokenlar)

# 2. Stemming - Porter
porter = PorterStemmer()
porter_result = [porter.stem(t) for t in temiz_tokenlar]
print('\nPorter Stemming :', porter_result)

# 3. Stemming - Snowball (İngilizce)
snowball = SnowballStemmer('english')
snowball_result = [snowball.stem(t) for t in temiz_tokenlar]
print('Snowball Stemming:', snowball_result)

# 4. Lemmatization - NLTK WordNet (POS etiketiyle daha doğru)
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(nltk_tag):
    """NLTK POS etiketini WordNet formatına çevirir."""
    if nltk_tag.startswith('J'): return wordnet.ADJ
    elif nltk_tag.startswith('V'): return wordnet.VERB
    elif nltk_tag.startswith('N'): return wordnet.NOUN
    elif nltk_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

pos_tagged = pos_tag(temiz_tokenlar)
lemma_result = [lemmatizer.lemmatize(w, get_wordnet_pos(pos))
                for w, pos in pos_tagged]
print('NLTK Lemmatization (POS ile):', lemma_result)

# 5. Karşılaştırma tablosu
print('\n{:<15} {:<15} {:<15} {:<15}'.format(
    'Orijinal', 'Porter', 'Snowball', 'Lemma(POS)'))
print('-' * 65)
for orig, port, snow, lemma in zip(temiz_tokenlar, porter_result,
                                    snowball_result, lemma_result):
    print(f'{orig:<15} {port:<15} {snow:<15} {lemma:<15}')

# 6. spaCy ile Lemmatization (Endüstri standardı)
try:
    nlp = spacy.load('en_core_web_sm')
    doc = nlp(metin)
    print('\n=== spaCy Lemmatization ===')
    print([(t.text, t.lemma_, t.pos_) for t in doc if not t.is_stop and t.is_alpha])
except:
    print('spaCy: python -m spacy download en_core_web_sm')


### 9.1.1.5. Tam Metin Normalizasyon Pipeline'ı

`bolum09/09_01_01_05_tam-metin-normalizasyon-pipeline-i.py`

_Kitap: Kod 9.4_


In [ ]:
import re
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

class MetinNormalizasyonPipeline:
    """
    Tam metin ön işleme pipeline'ı.
    Zincir: Temizlik → Tokenization → Stop-words → Stemming/Lemmatization
    """
    def __init__(self, dil='english', yontem='stemming', min_uzunluk=2):
        self.dil = dil
        self.yontem = yontem  # 'stemming' veya 'lemmatization'
        self.min_uzunluk = min_uzunluk
        self.stop_words = set(stopwords.words(dil))
        if yontem == 'stemming':
            from nltk.stem import SnowballStemmer
            self.normalizer = SnowballStemmer(dil)
        else:
            try:
                self.nlp = spacy.load('en_core_web_sm')
            except:
                print('spaCy modeli yok, stemming kullanılıyor.')
                from nltk.stem import SnowballStemmer
                self.normalizer = SnowballStemmer(dil)
                self.yontem = 'stemming'

    def on_temizlik(self, metin):
        metin = re.sub(r'http\S+|www.\S+', '', metin)  # URL
        metin = re.sub(r'@\w+|#\w+', '', metin)         # @mention, #hashtag
        metin = re.sub(r'[^a-zA-Z\s]', ' ', metin)      # Özel karakter
        return metin.lower().strip()

    def isle(self, metin):
        # Adım 1: Temizlik
        temiz = self.on_temizlik(metin)
        # Adım 2: Tokenization
        tokenlar = word_tokenize(temiz)
        # Adım 3: Stop-words filtresi + minimum uzunluk
        tokenlar = [t for t in tokenlar
                    if t not in self.stop_words and len(t) >= self.min_uzunluk]
        # Adım 4: Stemming veya Lemmatization
        if self.yontem == 'stemming':
            tokenlar = [self.normalizer.stem(t) for t in tokenlar]
        else:
            doc = self.nlp(' '.join(tokenlar))
            tokenlar = [t.lemma_ for t in doc]
        return tokenlar

    def isle_toplu(self, metinler):
        """Birden fazla metni toplu işler."""
        return [' '.join(self.isle(m)) for m in metinler]

# Kullanım
pipeline = MetinNormalizasyonPipeline(dil='english', yontem='stemming')

ornekler = [
    "The researchers are studying deep learning algorithms!",
    "NLP models have been improving rapidly since 2017. @user #AI",
    "She was running faster than the other players in the team."
]

print('=== Pipeline Çıktıları ===')
for ornek in ornekler:
    print(f'\nGiriş: {ornek}')
    print(f'Çıkış: {pipeline.isle(ornek)}')

# Toplu işleme (corpus için)
islenmiş_corpus = pipeline.isle_toplu(ornekler)
print('\n=== İşlenmiş Corpus (Modele Hazır) ===')
for m in islenmiş_corpus:
    print(m)


### 9.1.2.3. Python Uygulaması: Scikit-learn ile BoW ve TF-IDF

`bolum09/09_01_02_03_python-uygulamasi-scikit-learn-ile-bow-ve-tf-idf.py`

_Kitap: Kod 9.5_


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Örnek Belge Kümesi (Corpus)
corpus = [
    'veri madenciliği ile veri bilimi çok önemlidir',
    'makine öğrenmesi algoritmaları veri madenciliği temellidir',
    'derin öğrenme ve yapay zeka geleceği şekillendirir',
    'nlp metin madenciliği duygu analizi uygular',
    'metin sınıflandırma veri ön işleme gerektirir',
]

# ====================================================
# 1. BAG-OF-WORDS (Count Vectorizer)
# ====================================================
print('=' * 60)
print('BAG-OF-WORDS MATRİSİ')
print('=' * 60)

bow_vectorizer = CountVectorizer(
    min_df=1,          # En az 1 belgede geçmeli
    max_df=1.0,        # En fazla %100 belgede geçmeli (stop-word etkisi)
    ngram_range=(1, 2) # Unigram + Bigram
)
bow_matrix = bow_vectorizer.fit_transform(corpus)

print(f'Kelime Dağarcığı Boyutu (1-2 gram): {len(bow_vectorizer.vocabulary_)}')

# Sadece unigram için tekrar deneyelim
bow_uni = CountVectorizer(min_df=1)
bow_uni_matrix = bow_uni.fit_transform(corpus)

bow_df = pd.DataFrame(bow_uni_matrix.toarray(),
                       columns=bow_uni.get_feature_names_out(),
                       index=[f'Belge {i+1}' for i in range(len(corpus))])
print('\nBag-of-Words Matrisi (Unigram):')
print(bow_df)

# ====================================================
# 2. TF-IDF VEKTÖRİZASYONU
# ====================================================
print('\n' + '=' * 60)
print('TF-IDF MATRİSİ')
print('=' * 60)

tfidf_vectorizer = TfidfVectorizer(
    min_df=1,           # En az 1 belgede geçmeli
    max_df=0.95,        # Tüm belgelerde geçiyorsa kaldır (%95 üstü)
    sublinear_tf=True,  # TF için logaritmik ölçek: 1 + log(TF)
    norm='l2',          # L2 normalizasyon (kosinüs benzerliği için)
    ngram_range=(1, 1)  # Sadece unigram
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                         columns=tfidf_vectorizer.get_feature_names_out(),
                         index=[f'Belge {i+1}' for i in range(len(corpus))])
print('\nTF-IDF Matrisi:')
print(round(tfidf_df, 3))

# ====================================================
# 3. TF-IDF SKORLARI ANALİZİ
# ====================================================
print('\n=== Her Belge için En Ayırt Edici Kelimeler ===')
feature_names = tfidf_vectorizer.get_feature_names_out()
for i, belge in enumerate(corpus):
    tfidf_skorlar = tfidf_matrix[i].toarray()[0]
    top_idx = tfidf_skorlar.argsort()[-3:][::-1]  # En yüksek 3 skor
    top_kelimeler = [(feature_names[j], round(tfidf_skorlar[j], 4))
                     for j in top_idx if tfidf_skorlar[j] > 0]
    print(f'Belge {i+1}: {top_kelimeler}')

# ====================================================
# 4. KOSİNÜS BENZERLİĞİ ile Belge Benzerliği
# ====================================================
print('\n=== Belgeler Arası Kosinüs Benzerliği ===')
cos_sim = cosine_similarity(tfidf_matrix)
sim_df = pd.DataFrame(cos_sim,
                       index=[f'B{i+1}' for i in range(len(corpus))],
                       columns=[f'B{i+1}' for i in range(len(corpus))])
print(round(sim_df, 3))

# En benzer belge çifti
np.fill_diagonal(cos_sim, 0)  # Kendisiyle benzerliği sıfırla
max_idx = np.unravel_index(cos_sim.argmax(), cos_sim.shape)
print(f'\nEn benzer belge çifti: Belge {max_idx[0]+1} - Belge {max_idx[1]+1}')
print(f'Kosinüs Benzerliği: {cos_sim[max_idx]:.4f}')

# ====================================================
# 5. METIN SINIFLANDIRMA ile TF-IDF Entegrasyonu
# ====================================================
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Genişletilmiş etiketli corpus
belgeler = [
    'veri madenciliği veri analizi önemlidir',
    'makine öğrenmesi sınıflandırma algoritmaları',
    'derin öğrenme sinir ağları yapay zeka',
    'veri bilimi veri temizleme önişleme',
    'nlp metin işleme dil modeli',
    'kümeleme boyut indirgeme PCA algoritmaları',
    'regresyon sınıflandırma karar ağacı',
    'doğal dil işleme transformer BERT',
]
etiketler = [0, 0, 1, 0, 1, 0, 0, 1]  # 0: Veri Mad., 1: NLP/Derin Öğrenme

X_train, X_test, y_train, y_test = train_test_split(
    belgeler, etiketler, test_size=0.25, random_state=42
)

# TF-IDF + Naive Bayes Pipeline
clf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=1, sublinear_tf=True)),
    ('clf', MultinomialNB(alpha=0.1))
])

clf_pipeline.fit(X_train, y_train)
y_pred = clf_pipeline.predict(X_test)

print('\n=== TF-IDF + Naive Bayes Sınıflandırma ===')
print(classification_report(y_test, y_pred,
      target_names=['Veri Mad.', 'NLP/DL'], zero_division=0))


## 9.2. Modern NLP: Anlamsal Analiz


### 9.2.1.7. Python Uygulaması: Gensim ile Word2Vec Eğitimi ve Analizi

`bolum09/09_02_01_07_python-uygulamasi-gensim-ile-word2vec-egitimi-ve.py`

_Kitap: Kod 9.6_


In [ ]:
import nltk
import numpy as np
from gensim.models import Word2Vec, KeyedVectors
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ============================================================
# 1. CORPUS VE EĞİTİM
# ============================================================
cumleler = [
    'kral sarayda hüküm sürer ve ülkeyi yönetir',
    'kraliçe sarayda oturur ve ülkeyi yönetir',
    'erkek adam güçlü ve cesur olmalıdır',
    'kadın güçlü ve cesur olabilir kraliçe gibi',
    'kedi evcil hayvan olarak evde yaşar',
    'köpek evcil hayvan olarak evde yaşar',
    'kedi ve köpek çok iyi evcil hayvanlardır',
    'veri bilimi makine öğrenmesi algoritmaları ile çalışır',
    'yapay zeka derin öğrenme sinir ağları kullanır',
    'doğal dil işleme metin analizi için kullanılır',
    'python programlama dili veri analizi için idealdir',
    'java programlama dili nesne yönelimli bir dildir',
    'paris fransa başkentidir avrupa şehridir',
    'berlin almanya başkentidir avrupa şehridir',
    'roma italya başkentidir tarihi avrupa şehridir',
]

# Tokenize: Cümleleri kelime listelerine böl
token_cumleler = [nltk.word_tokenize(c.lower()) for c in cumleler]

# Word2Vec modeli eğitimi
model = Word2Vec(
    sentences=token_cumleler,
    vector_size=50,    # Embedding boyutu (üretimde 100-300)
    window=3,          # Bağlam penceresi yarıçapı
    min_count=1,       # Minimum kelime frekansı
    sg=1,              # 1=Skip-Gram, 0=CBOW
    negative=5,        # Negative sampling sayısı
    epochs=100,        # Eğitim döngüsü sayısı
    workers=4          # Paralel iş parçacığı
)

print(f'Kelime Dağarcığı: {len(model.wv)} kelime')
print(f'Vektör Boyutu: {model.vector_size}')

# ============================================================
# 2. KELİME VEKTÖRLERİ VE BENZERLİK
# ============================================================
print('\n=== Kelime Vektörü (kral) — İlk 10 boyut ===')
print(np.round(model.wv['kral'][:10], 4))

print('\n=== kedi kelimesine en benzer kelimeler ===')
for kelime, skor in model.wv.most_similar('kedi', topn=5):
    print(f'  {kelime:<15} benzerlik: {skor:.4f}')

print('\n=== kral-erkek+kadın analojisi ===')
try:
    sonuc = model.wv.most_similar(
        positive=['kral', 'kadın'],
        negative=['erkek'],
        topn=3
    )
    print('Beklenen: kraliçe')
    for k, s in sonuc:
        print(f'  {k}: {s:.4f}')
except Exception as e:
    print(f'Küçük corpus sınırlaması: {e}')

# Kosinüs benzerliği hesaplama
if 'kedi' in model.wv and 'köpek' in model.wv:
    sim_kedi_kopek = model.wv.similarity('kedi', 'köpek')
    sim_kedi_ucak = model.wv.similarity('kedi', 'veri')
    print(f'\nkedi — köpek benzerliği: {sim_kedi_kopek:.4f}')
    print(f'kedi — veri benzerliği:  {sim_kedi_ucak:.4f}')

# ============================================================
# 3. MODELİ KAYDETME VE YÜKLEME
# ============================================================
model.wv.save('kelime_vektorleri.kv')
# Yükleme:
# loaded_wv = KeyedVectors.load('kelime_vektorleri.kv')
print('\nModel kaydedildi: kelime_vektorleri.kv')

# ============================================================
# 4. PCA İLE 2 BOYUTLU GÖRSELLEŞTİRME
# ============================================================
kelimeler = ['kral', 'kraliçe', 'erkek', 'kadın', 'kedi', 'köpek',
             'paris', 'berlin', 'roma', 'python', 'java']
mevcut = [k for k in kelimeler if k in model.wv]

if mevcut:
    vektorler = np.array([model.wv[k] for k in mevcut])
    pca = PCA(n_components=2)
    koordinatlar = pca.fit_transform(vektorler)

    plt.figure(figsize=(10, 7))
    plt.scatter(koordinatlar[:, 0], koordinatlar[:, 1],
                c='steelblue', s=100, alpha=0.7)
    for i, kelime in enumerate(mevcut):
        plt.annotate(kelime, (koordinatlar[i, 0], koordinatlar[i, 1]),
                     fontsize=12, ha='right')
    plt.title('Word2Vec Vektörlerinin PCA ile 2D Projeksiyonu')
    plt.xlabel('1. Bileşen')
    plt.ylabel('2. Bileşen')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('w2v_pca.png', dpi=150)
    plt.show()
    print(f'\nPCA Açıklanan Varyans: {pca.explained_variance_ratio_.sum():.2%}')


### 9.2.1.8. Hazır Önceden Eğitilmiş Modeller ile Transfer Öğrenmesi

`bolum09/09_02_01_08_hazir-onceden-egitilmis-modeller-ile-transfer-og.py`

_Kitap: Kod 9.7_


In [ ]:
import gensim.downloader as api
from gensim.models import KeyedVectors
import numpy as np

# ============================================================
# 1. GENSIM'DEN HAZIR MODEL (İNDİRİLMESİ GEREKİR)
# ============================================================
# Mevcut modelleri listele
print('Kullanılabilir hazır modeller:')
for model_name in list(api.info()['models'].keys())[:10]:
    print(f'  - {model_name}')

# Google News Word2Vec (3 milyar kelime, 300 boyut) — 1.6 GB
# wv = api.load('word2vec-google-news-300')

# GloVe Wikipedia + Gigaword (6B kelime, 100 boyut) — 128 MB
# wv = api.load('glove-wiki-gigaword-100')

# Küçük test modeli (hızlı indirme)
print('\nKüçük model yükleniyor...')
wv = api.load('glove-wiki-gigaword-50')
print(f'Model yüklendi: {len(wv)} kelime, {wv.vector_size} boyut')

# ============================================================
# 2. HAZIR MODEL ANALİZLERİ
# ============================================================
print('\n=== king - man + woman analojisi ===')
result = wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
for word, score in result:
    print(f'  {word}: {score:.4f}')
# Beklenen çıktı: queen ≈ 0.85

print('\n=== Ülke - Başkent analojisi (Paris - France + Germany) ===')
result = wv.most_similar(positive=['paris', 'germany'], negative=['france'], topn=3)
for word, score in result:
    print(f'  {word}: {score:.4f}')
# Beklenen: berlin

# ============================================================
# 3. EMBEDDING'İ ÖZELLİK OLARAK KULLANMA (Metin Sınıflandırma)
# ============================================================
def metin_vektoru(metin, wv, boyut=50):
    """
    Bir metindeki tüm kelimelerin embedding'lerinin ortalamasını
    alarak metin düzeyinde vektör üretir.
    """
    kelimeler = metin.lower().split()
    vektorler = [wv[k] for k in kelimeler if k in wv]
    if not vektorler:
        return np.zeros(boyut)
    return np.mean(vektorler, axis=0)

# Test
metinler = [
    'the movie was absolutely fantastic and entertaining',
    'terrible film complete waste of time',
    'neural networks learn from large datasets',
]
for m in metinler:
    vec = metin_vektoru(m, wv)
    print(f'\nMetin: {m[:40]}...')
    print(f'Vektör şekli: {vec.shape}, İlk 5 değer: {np.round(vec[:5], 4)}')

# ============================================================
# 4. HUGGINGFACE İLE BERT EMBEDDING'İ
# ============================================================
try:
    from transformers import AutoTokenizer, AutoModel
    import torch

    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    bert_model = AutoModel.from_pretrained('bert-base-uncased')

    metin = 'The bank is on the river bank near the financial bank'
    inputs = tokenizer(metin, return_tensors='pt', padding=True)

    # [CLS] token embedding — metin düzeyi temsil
    with torch.no_grad():
        outputs = bert_model(**inputs)

    cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()
    print(f'\nBERT [CLS] embedding boyutu: {cls_embedding.shape}')
    # Her 'bank' kelimesi için farklı bağlamsal vektör üretildi!
except ImportError:
    print('Kurulum: pip install transformers torch')


### 9.2.2.4. Python Uygulaması I: VADER ile Sosyal Medya Analizi

`bolum09/09_02_02_04_python-uygulamasi-i-vader-ile-sosyal-medya-anali.py`

_Kitap: Kod 9.8_


In [ ]:
import nltk
import pandas as pd
import numpy as np
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from collections import Counter

nltk.download('vader_lexicon', quiet=True)
analyzer = SentimentIntensityAnalyzer()

# ============================================================
# 1. TEMEL VADER ANALİZİ
# ============================================================
ornekler = [
    # Normal pozitif/negatif
    'The new product is absolutely fantastic! Highly recommend.',
    'Terrible service, worst experience of my life.',
    'The movie was okay, nothing special.',
    # Büyük harf şiddetlendirme
    'This is AMAZING! Best thing ever!!!',
    'This is amazing. Best thing ever.',
    # Negasyon testi
    'The food is not good at all.',
    'The food is good.',
    # Sarkasm (VADER bu konuda sınırlı)
    'Oh great, another Monday morning...',
    # Emoji ve sosyal medya
    'Just got promoted! :D #blessed',
    'Stuck in traffic again... :(',
]

print('='*70)
print(f'{"Tweet":<45} {"Neg":>5} {"Pos":>5} {"Comp":>6} {"Karar"}')
print('='*70)

for tweet in ornekler:
    s = analyzer.polarity_scores(tweet)
    karar = 'OLUMLU' if s['compound'] >= 0.05 else (
             'OLUMSUZ' if s['compound'] <= -0.05 else 'NÖTR')
    print(f"{tweet[:44]:<45} {s['neg']:>5.2f} {s['pos']:>5.2f}"
          f" {s['compound']:>6.3f} {karar}")

# ============================================================
# 2. SOSYAL MEDYA CORPUS ANALİZİ
# ============================================================
sosyal_medya_df = pd.DataFrame({
    'id': range(1, 11),
    'platform': ['Twitter']*5 + ['Reddit']*5,
    'metin': [
        'Love the new update! Works perfectly now #tech',
        'App keeps crashing! Developers fix this ASAP',
        'Decent product for the price, nothing extraordinary',
        'WORST APP EVER!!! Delete immediately!!!',
        'Pretty good actually, surprised by the quality :)',
        'This is the best library I have used in years!',
        'Does not work as advertised, very disappointing',
        'Average at best, would not buy again',
        'Outstanding! Exceeded all my expectations',
        'Terrible quality, broke after one day',
    ]
})

# VADER analizi uygula
def vader_analiz(metin):
    s = analyzer.polarity_scores(metin)
    s['karar'] = ('OLUMLU' if s['compound'] >= 0.05 else
                  'OLUMSUZ' if s['compound'] <= -0.05 else 'NÖTR')
    return pd.Series(s)

sonuclar = sosyal_medya_df['metin'].apply(vader_analiz)
df_analiz = pd.concat([sosyal_medya_df, sonuclar], axis=1)

print('\n=== Sosyal Medya Duygu Analizi Sonuçları ===')
print(df_analiz[['platform', 'metin', 'compound', 'karar']].to_string())

# ============================================================
# 3. PLATFORM VE DUYGU DAĞILIMI
# ============================================================
print('\n=== Platform Bazlı Duygu Dağılımı ===')
print(df_analiz.groupby(['platform', 'karar']).size().unstack(fill_value=0))

print('\n=== Ortalama Compound Skorları ===')
print(df_analiz.groupby('platform')['compound'].agg(['mean', 'std', 'min', 'max']))

# ============================================================
# 4. ZAMAN SERİSİ TREND ANALİZİ
# ============================================================
import random
random.seed(42)

# Simüle edilmiş günlük tweet verisi
gunler = pd.date_range('2024-01-01', periods=30)
gunluk_duygu = []

for gun in gunler:
    gun_skorlari = [random.gauss(0.1, 0.5) for _ in range(50)]
    gun_skorlari = [max(-1, min(1, s)) for s in gun_skorlari]
    gunluk_duygu.append({
        'tarih': gun,
        'ortalama_compound': np.mean(gun_skorlari),
        'pozitif_oran': sum(1 for s in gun_skorlari if s >= 0.05) / 50,
        'negatif_oran': sum(1 for s in gun_skorlari if s <= -0.05) / 50,
    })

trend_df = pd.DataFrame(gunluk_duygu)
print('\n=== 30 Günlük Duygu Trend Özeti ===')
print(trend_df[['tarih','ortalama_compound','pozitif_oran','negatif_oran']].tail(5))


### 9.2.2.5. Python Uygulaması II: Makine Öğrenmesi ile Duygu Sınıflandırma

`bolum09/09_02_02_05_python-uygulamasi-ii-makine-ogrenmesi-ile-duygu.py`

_Kitap: Kod 9.9_


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. ETİKETLENMİŞ VERİ SETİ
# ============================================================
belgeler = [
    # OLUMLU
    'This product is absolutely amazing quality is superb',
    'Excellent service fast delivery highly recommend',
    'Best purchase I have made great value for money',
    'Outstanding performance exceeded my expectations',
    'Fantastic product works perfectly love it',
    'Very happy with this purchase great quality',
    'Wonderful experience customer service is brilliant',
    'Perfect exactly what I needed works great',
    # OLUMSUZ
    'Terrible quality broke after two days waste of money',
    'Absolutely horrible product do not buy this',
    'Worst purchase ever completely useless',
    'Very disappointed quality is poor not worth the price',
    'Awful experience product stopped working immediately',
    'Completely broken arrived damaged not recommended',
    'Poor quality fails to deliver on promises',
    'Dreadful product returned it immediately',
    # NÖTR
    'The product works as described nothing special',
    'Average quality for the price acceptable',
    'Decent product does what it says on the box',
    'Okay product nothing extraordinary',
    'It works fine not great but not bad either',
    'Standard quality meets basic requirements',
]
etiketler = (['olumlu']*8 + ['olumsuz']*8 + ['nötr']*6)

X_train, X_test, y_train, y_test = train_test_split(
    belgeler, etiketler,
    test_size=0.25,
    random_state=42,
    stratify=etiketler
)

# ============================================================
# 2. ÇOKLU MODEL KARŞILAŞTIRMASI
# ============================================================
modeller = {
    'Multinomial Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=1, sublinear_tf=True,
                                   ngram_range=(1,2))),
        ('clf', MultinomialNB(alpha=0.5))
    ]),
    'Complement Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=1, sublinear_tf=True,
                                   ngram_range=(1,2))),
        ('clf', ComplementNB(alpha=0.5))
    ]),
    'Logistik Regresyon': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=1, sublinear_tf=True,
                                   ngram_range=(1,2))),
        ('clf', LogisticRegression(max_iter=1000, C=1.0))
    ]),
    'LinearSVC': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=1, sublinear_tf=True,
                                   ngram_range=(1,2))),
        ('clf', LinearSVC(C=1.0, max_iter=2000))
    ]),
}

print('=== Model Karşılaştırması (5 Katlı Çapraz Doğrulama) ===')
print(f'{"Model":<25} {"CV F1 Ort.":>12} {"±Std":>8}')
print('-' * 48)

en_iyi_model = None
en_iyi_f1 = 0

for model_adi, pipeline in modeller.items():
    cv_scores = cross_val_score(
        pipeline, belgeler, etiketler,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='f1_weighted'
    )
    print(f'{model_adi:<25} {cv_scores.mean():>12.4f} {cv_scores.std():>8.4f}')
    if cv_scores.mean() > en_iyi_f1:
        en_iyi_f1 = cv_scores.mean()
        en_iyi_model = (model_adi, pipeline)

# ============================================================
# 3. EN İYİ MODEL — DETAYLI DEĞERLENDİRME
# ============================================================
print(f'\n=== En İyi Model: {en_iyi_model[0]} ===')
en_iyi_model[1].fit(X_train, y_train)
y_pred = en_iyi_model[1].predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))

# ============================================================
# 4. YENI METİNLER İÇİN TAHMİN
# ============================================================
yeni_metinler = [
    'Absolutely love this! Best product I have ever used!',
    'Complete garbage, broke on day one, do not buy',
    'It does the job, nothing to write home about',
    'NOT happy with this at all, very disappointed!',
]

print('\n=== Yeni Metin Tahminleri ===')
tahminler = en_iyi_model[1].predict(yeni_metinler)
for m, t in zip(yeni_metinler, tahminler):
    print(f'  [{t.upper():>8}] {m[:55]}')

# ============================================================
# 5. EN AYIRT EDİCİ KELİMELER
# ============================================================
# Pipeline içinden TF-IDF'e eriş
pipeline = en_iyi_model[1]
if hasattr(pipeline['clf'], 'coef_'):
    feature_names = pipeline['tfidf'].get_feature_names_out()
    siniflar = pipeline['clf'].classes_
    print('\n=== Sınıfa Göre En Ayırt Edici Kelimeler (Top 5) ===')
    for i, sinif in enumerate(siniflar):
        if hasattr(pipeline['clf'], 'coef_'):
            top_idx = np.argsort(pipeline['clf'].coef_[i])[-5:][::-1]
            top_kelimeler = [(feature_names[j], round(pipeline['clf'].coef_[i][j], 3))
                             for j in top_idx]
            print(f'{sinif.upper():>10}: {top_kelimeler}')


### 9.2.2.6. Aspekt Tabanlı Duygu Analizi (ABSA)

`bolum09/09_02_02_06_aspekt-tabanli-duygu-analizi.py`

_Kitap: Kod 9.10_


In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
analyzer = SentimentIntensityAnalyzer()

# Basit kural tabanlı ABSA demo
# Gerçek ABSA için fine-tuned BERT modelleri kullanılır

ASPEKT_SOZLUGU = {
    'kamera': ['kamera', 'fotoğraf', 'çekim', 'lens', 'zoom', 'görüntü'],
    'pil': ['pil', 'batarya', 'şarj', 'enerji', 'dayanma'],
    'fiyat': ['fiyat', 'para', 'ücret', 'maliyet', 'değer', 'ucuz', 'pahalı'],
    'tasarim': ['tasarım', 'görünüm', 'hafif', 'ağır', 'güzel', 'estetik'],
    'performans': ['hız', 'işlemci', 'yavaş', 'hızlı', 'kasma', 'performans'],
}

def aspekt_duygu_analiz(yorum):
    """
    Verilen yorumdan aspektleri tespit eder ve
    her aspekt için duygu skoru hesaplar.
    """
    yorum_lower = yorum.lower()
    cumleler = yorum.split(',')  # Basit cümle bölme
    sonuclar = {}

    for aspekt, kelimeler in ASPEKT_SOZLUGU.items():
        ilgili_cumleler = []
        for cumle in cumleler:
            cumle_lower = cumle.lower()
            if any(k in cumle_lower for k in kelimeler):
                ilgili_cumleler.append(cumle.strip())

        if ilgili_cumleler:
            # Aspektle ilgili cümlelerin VADER skorunu hesapla
            skorlar = [analyzer.polarity_scores(c)['compound']
                       for c in ilgili_cumleler]
            ortalama_skor = sum(skorlar) / len(skorlar)
            karar = ('Pozitif' if ortalama_skor >= 0.05 else
                     'Negatif' if ortalama_skor <= -0.05 else 'Nötr')
            sonuclar[aspekt] = {
                'skor': round(ortalama_skor, 3),
                'karar': karar,
                'ilgili_cumle': ' | '.join(ilgili_cumleler)
            }

    return sonuclar

# Test
yorumlar = [
    'Kamera kalitesi mükemmel, fotoğraflar çok net. Fakat pil ömrü berbat, yarım günde bitiyor.',
    'Fiyatı biraz pahalı ama performansı gerçekten iyi. Tasarım da şık görünüyor.',
    'Hız açısından harika, kasma yok. Şarj çok uzun sürüyor ve pil zayıf.',
]

for yorum in yorumlar:
    print(f'\nYorum: {yorum[:60]}...')
    print('Aspekt Analizi:')
    sonuclar = aspekt_duygu_analiz(yorum)
    for aspekt, bilgi in sonuclar.items():
        print(f"  {aspekt:>12}: {bilgi['karar']:>8} (skor: {bilgi['skor']:>6}) — {bilgi['ilgili_cumle'][:40]}")
